<a href="https://colab.research.google.com/github/RodolfoFerro/beepy5/blob/main/notebooks/Notebook_1_La_imagen_como_dato.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Taller] **Imágenes como datos:** redes neuronales y visión por computadora 🔬

> **Descripción:** En este taller abordaremos el procesamiento de imágenes como un problema de aprendizaje automático, explorando cómo las redes neuronales artificiales procesan y comprenden imágenes, extrayendo patrones y características que escapan a la percepción humana. Usando Python, implementaremos un pipeline completo (desde la imagen cruda hasta los resultados) y visualizaremos qué aprenden los modelos, convirtiendo la "caja negra" en algo comprensible e interpretable. <br>
> **Autor:** Rodolfo Ferro <br>
> **Contacto:** ferro@cimat.mx / [@rodo_ferro](https://www.instagram.com/rodo_ferro/)

## **Notebook 1** - La imagen como dato
---

**Duración estimada:** ~30 minutos  
**Dataset:** BloodMNIST — imágenes de células sanguíneas (28×28 px, 8 clases)  

Este notebook acompaña la presentación teórica. Cada sección corresponde a un bloque de slides.

**IMPORTANTE:** La idea es **ejecutar y observar** — no memorizar código.

### Clases del dataset

| ID | Tipo de célula | Descripción breve |
|----|---------------|-------------------|
| 0  | Basófilo | Granulocito poco frecuente, gránulos morados |
| 1  | Eosinófilo | Granulocito con gránulos rojizos |
| 2  | Eritroblasto | Precursor de glóbulo rojo, núcleo oscuro |
| 3  | Linfocito (IG) | Linfocito granular grande |
| 4  | Linfocito | Célula inmune de núcleo redondo y oscuro |
| 5  | Monocito | El leucocito más grande, núcleo arriñonado |
| 6  | Neutrófilo | El más abundante; núcleo multilobulado |
| 7  | Plaqueta | Fragmento celular diminuto, sin núcleo |

---

El siguiente bloque de código instalará las dependencias a utilizar.

In [ ]:
# Instalación de dependencias

# Ejecutar solo una vez al inicio de la sesión de Colab
!pip install -q medmnist timm scikit-image
print("✅ Dependencias listas")

Comenzaremos importando las paqueterías que nos serán de utilizad a lo largo del cuaderno.

In [ ]:
# Imports globales

# Objetos numéricos y gráficos
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

# Dataset a utilizar
import medmnist
from medmnist import BloodMNIST, INFO

# Procesamiento de imágenes
import skimage.color as skcolor
import skimage.filters as skfilters
import skimage.feature as skfeature
from scipy.ndimage import convolve

# Modelos / Redes neuronales
import torch
import timm
import torchvision.transforms as T

# Estilo de gráficas (p/plt)
plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor": "#0d1117",
    "axes.edgecolor": "#444",
    "axes.labelcolor": "#aaa",
    "xtick.color": "#aaa",
    "ytick.color": "#aaa",
    "text.color": "white",
    "image.cmap": "viridis",
})

NOMBRES_CLASE = [
    "Basófilo",
    "Eosinófilo",
    "Eritroblasto",
    "Linfocito (IG)",
    "Linfocito",
    "Monocito",
    "Neutrófilo",
    "Plaqueta"
]

print("✅ Imports listos")
print(f"   medmnist {medmnist.__version__}  |  torch {torch.__version__}")

---
## **Sección 0** — Carga de datos

In [ ]:
# Descarga de BloodMNIST

# split="train" descarga ~11K imágenes de entrenamiento
# as_rgb=True garantiza 3 canales (algunas imágenes de MedMNIST pueden ser grises)
dataset_train = BloodMNIST(split="train", download=True, as_rgb=True)
dataset_test  = BloodMNIST(split="test",  download=True, as_rgb=True)

print()
print(f"Imágenes de entrenamiento : {len(dataset_train)}")
print(f"Imágenes de prueba        : {len(dataset_test)}")
print(f"Clases                    : {len(NOMBRES_CLASE)}")

# Extrae arrays numpy para manipulación directa
# Cada elemento del dataset es (PIL_Image, label_array)
imagenes = np.array([
    np.array(dataset_train[i][0]) for i in range(len(dataset_train))
])
etiquetas = np.array([
    dataset_train[i][1][0] for i in range(len(dataset_train))
])

print()
print(f"Shape del array completo  : {imagenes.shape} - (N, H, W, C)")
print(f"Dtype                     : {imagenes.dtype}")
print(f"Rango de valores          : [{imagenes.min()}, {imagenes.max()}]")

---
## **Sección 1** — La imagen como dato

Una imagen digital es simplemente un arreglo tridimensional de números:

$$I \in \mathbb{R}^{H \times W \times C}$$

Para BloodMNIST: $H = W = 28$, $C = 3$ (canales R, G, B).

Empecemos por entender qué hay realmente dentro de una sola imagen.

In [ ]:
# Exploramos una imagen individual

# Tomamos un neutrófilo (clase 6) como ejemplo
idx = np.where(etiquetas == 6)[0][0]
img = imagenes[idx]  # shape: (28, 28, 3), dtype: uint8

print(f"Célula: {NOMBRES_CLASE[6]}")
print(f"Shape : {img.shape}  -  ({img.shape[0]} filas × {img.shape[1]} cols × {img.shape[2]} canales)")
print(f"Dtype : {img.dtype}")
print(f"Min   : {img.min()}   Max: {img.max()}   Media: {img.mean():.1f}")
print()
print("Esquina superior izquierda (5×5 píxeles, canal R):")
print(img[:5, :5, 0])  # Canal R solamente

Para visualizar un canal de color, podemos hacer 0 el resto de los canales y sólo asignar los valores del canal de interés dentro de la imagen.

In [ ]:
# Visualicemos un canal de color

# Canal rojo
canal_rojo = np.zeros(img.shape, dtype="uint8")
canal_rojo[:, :, 0] = img[:, :, 0]
plt.imshow(canal_rojo)
plt.axis("off")
plt.show()

Ahora tú visualiza el canal verde y el canal azul.

In [ ]:
# Canal verde


# Canal azul


In [ ]:
# Visualizar la imagen con sus tres canales de color

# R
canal_rojo = np.zeros(img.shape, dtype="uint8")
canal_rojo[:, :, 0] = img[:, :, 0]

# G
canal_verde = np.zeros(img.shape, dtype="uint8")
canal_verde[:, :, 1] = img[:, :, 1]

# B
canal_azul = np.zeros(img.shape, dtype="uint8")
canal_azul[:, :, 2] = img[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(12, 3), dpi=300)
titulos   = [
    "RGB (original)",
    "Canal R (rojo)",
    "Canal G (verde)",
    "Canal B (azul)"
]

for ax, titulo, canal in zip(
        axes,
        titulos,
        [img, canal_rojo, canal_verde, canal_azul]):
    ax.imshow(canal, interpolation="nearest")
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")

fig.suptitle(f"{NOMBRES_CLASE[6]} — descomposición RGB", y=1.02)
plt.tight_layout()
plt.show()

# Estadísticas por canal
print()
print("Estadísticas por canal (RGB)")
print("="*30)
print(f"  R — media: {img[:,:,0].mean():.1f}  std: {img[:,:,0].std():.1f}")
print(f"  G — media: {img[:,:,1].mean():.1f}  std: {img[:,:,1].std():.1f}")
print(f"  B — media: {img[:,:,2].mean():.1f}  std: {img[:,:,2].std():.1f}")

In [ ]:
# Visualizar las 8 clases del dataset
fig, axes = plt.subplots(2, 4, figsize=(11, 6), dpi=300)

for clase_id, ax in enumerate(axes.flat):
    idx_clase = np.where(etiquetas == clase_id)[0][0]
    ax.imshow(imagenes[idx_clase], interpolation="nearest")
    ax.set_title(f"{clase_id}: {NOMBRES_CLASE[clase_id]}", fontsize=8)
    ax.axis("off")

fig.suptitle("Las 8 clases de BloodMNIST", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 🏋️ **Ejercicio 1**

Cambia `clase_id` en la celda siguiente para explorar distintas células.

Completa el código y observa los valores numéricos de cada canal.

**Pregunta:** ¿Qué canal (R, G o B) crees que captura mejor la distinción entre el núcleo y el citoplasma de un linfocito? ¿Por qué?

In [ ]:
# Ejercicio 1: Explora una célula a tu elección
clase_id = 4  # Cambia este número (0-7) y vuelve a ejecutar

idx_ej = np.where(etiquetas == clase_id)[0][2]  # Tercera instancia de la clase
img_ej = imagenes[idx_ej]

# Definimos los canales
# R
rojo_ej = np.zeros(img_ej.shape, dtype="uint8")
rojo_ej[:, :, 0] = None

# G
verde_ej = np.zeros(img_ej.shape, dtype="uint8")
verde_ej[:, :, 1] = None

# B
azul_ej = np.zeros(img_ej.shape, dtype="uint8")
azul_ej[:, :, 2] = None

fig, axes = plt.subplots(1, 4, figsize=(12, 3), dpi=300)
for ax, titulo, canal in zip(
        axes,
        ["RGB", "Canal R", "Canal G", "Canal B"],
        [img_ej, rojo_ej, verde_ej, azul_ej]):
    ax.imshow(canal, interpolation="nearest")
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")

fig.suptitle(f"Ejercicio 1 — {NOMBRES_CLASE[clase_id]}", y=1.02)
plt.tight_layout()
plt.show()

---
## **Sección 2** — Espacios de color

RGB no es el único modo de representar color. Dependiendo de la tarea, otros espacios pueden separar mejor la información relevante:

- **HSV** (Hue, Saturation, Value): separa el tono del brillo. Útil para filtrar por color de tinción.
- **LAB** (CIE L\*a\*b\*): diseñado para ser perceptualmente uniforme. Estándar en imágenes médicas.

La conversión es una transformación matemática del vector de 3 números — los píxeles no cambian, solo su *representación*.

In [ ]:
# Convertir una célula a distintos espacios de color

# skimage trabaja con imágenes float en [0, 1]
img_float = img.astype(np.float64) / 255.0
img_r = np.zeros(img_float.shape); img_r[:, :, 0] = img_float[:, :, 0]
img_g = np.zeros(img_float.shape); img_g[:, :, 1] = img_float[:, :, 1]
img_b = np.zeros(img_float.shape); img_b[:, :, 2] = img_float[:, :, 2]


img_hsv = skcolor.rgb2hsv(img_float)   # H ∈ [0, 1], S ∈ [0, 1], V ∈ [0, 1]
img_lab = skcolor.rgb2lab(img_float)   # L ∈ [0, 100], a ∈ [-128, 127], b ∈ [-128, 127]

print("Forma y rango de cada espacio:")
print(f"🌀 RGB  shape={img_float.shape}  rango=[{img_float.min():.2f}, {img_float.max():.2f}]")
print(f"🌀 HSV  shape={img_hsv.shape}  rango=[{img_hsv.min():.2f}, {img_hsv.max():.2f}]")
print(f"🌀 LAB  shape={img_lab.shape}  rango=[{img_lab.min():.2f}, {img_lab.max():.2f}]")

In [ ]:
# Visualizamos los canales de cada espacio
espacios = {
    "RGB": {
        "imagen": img_float,
        "canales": [img_r, img_g, img_b],
        "nombres": ["R (rojo)", "G (verde)", "B (azul)"],
        "cmaps":   ["gray"] * 3,
    },
    "HSV": {
        "imagen": skcolor.hsv2rgb(img_hsv),
        "canales": [img_hsv[:, :, 0], img_hsv[:, :, 1], img_hsv[:, :, 2]],
        "nombres": ["H (tono)", "S (saturación)", "V (valor/brillo)"],
        "cmaps":   ["hsv", "gray", "gray"],
    },
    "LAB": {
        "imagen": img_float,
        "canales": [img_lab[:, :, 0], img_lab[:, :, 1], img_lab[:, :, 2]],
        "nombres": ["L* (luminosidad)", "a* (verde↔rojo)", "b* (azul↔amarillo)"],
        "cmaps":   ["gray", "RdYlGn_r", "RdYlBu_r"],
    },
}

fig, axes = plt.subplots(3, 4, figsize=(13, 9), dpi=300)

for fila, (nombre_espacio, datos) in enumerate(espacios.items()):
    # Primera columna: imagen de referencia
    axes[fila, 0].imshow(datos["imagen"], interpolation="nearest")
    axes[fila, 0].set_title(f"{nombre_espacio}\n(referencia)", fontsize=9)
    axes[fila, 0].axis("off")
    axes[fila, 0].set_ylabel(nombre_espacio, fontsize=10, color="white")

    # Columnas 1-3: canales individuales
    for col, (canal, nombre_c, cmap) in enumerate(
            zip(datos["canales"], datos["nombres"], datos["cmaps"])):
        im = axes[fila, col+1].imshow(canal, cmap=cmap, interpolation="nearest")
        axes[fila, col+1].set_title(nombre_c, fontsize=9)
        axes[fila, col+1].axis("off")
        plt.colorbar(im, ax=axes[fila, col+1], fraction=0.046, pad=0.04)

fig.suptitle(f"Espacios de color — {NOMBRES_CLASE[6]}", fontsize=12)
plt.tight_layout()
plt.show()

### 🏋️ **Ejercicio 2** — Segmentación por umbral en L\*

El canal **L\*** de LAB codifica la luminosidad pura, independientemente del color. Las células con tinción de Giemsa tienen el núcleo más oscuro (L\* bajo) que el citoplasma.

Vamos a usar un umbral simple sobre L\* para intentar separar el núcleo del fondo.

In [ ]:
# Ejercicio 2: Segmentación por umbral en L*
umbral = 60   # Ajusta este valor (0-100) y observa el resultado

L_star  = img_lab[:, :, 0]           # Canal de luminosidad
mascara = L_star < umbral            # True donde la célula es oscura (núcleo)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), dpi=300)

axes[0].imshow(img_float, interpolation="nearest")
axes[0].set_title("Original (RGB)", fontsize=9)

axes[1].imshow(L_star, cmap="gray", interpolation="nearest")
axes[1].set_title(f"Canal L* (luminosidad)", fontsize=9)

axes[2].imshow(mascara, cmap="gray", interpolation="nearest")
axes[2].set_title(f"Máscara (L* < {umbral})", fontsize=9)

# Aplicar máscara sobre imagen original
img_segmentada = img_float.copy()
img_segmentada[~mascara] = [0.15, 0.15, 0.15]   # fondo gris oscuro
axes[3].imshow(img_segmentada, interpolation="nearest")
axes[3].set_title("Región segmentada", fontsize=9)

for ax in axes:
    ax.axis("off")

fig.suptitle(f"Segmentación por umbral — {NOMBRES_CLASE[6]}  (umbral L*={umbral})", y=1.02)
plt.tight_layout()
plt.show()

print(f"Píxeles clasificados como núcleo: {mascara.sum()} / {mascara.size} ({100*mascara.mean():.1f}%)")
print("Ajusta el `umbral` y vuelve a ejecutar. ¿Qué pasa si sube a 80? ¿Y si baja a 40?")

---
## **Sección 3** — El problema de la variabilidad

La misma célula fotografiada en condiciones distintas puede ser **muy diferente en píxeles**. Esto es un problema si queremos compararlas numéricamente.

Vamos a cuantificar qué tan mal funciona la distancia pixel a pixel como medida de similitud.

In [ ]:
# Aplicamos variaciones realistas a una imagen de célula

def variar_imagen(img_uint8, brillo=0, ruido_std=0, contraste=1.0):
    """Simula variaciones de captura: iluminación, sensor, magnificación."""
    img_f = img_uint8.astype(np.float32)
    img_f = img_f * contraste + brillo
    if ruido_std > 0:
        img_f += np.random.normal(0, ruido_std, img_f.shape)
    return np.clip(img_f, 0, 255).astype(np.uint8)

np.random.seed(42)
base = imagenes[np.where(etiquetas == 6)[0][0]]   # Neutrófilo base

variaciones = {
    "Original":             base,
    "Oscura (−60)":         variar_imagen(base, brillo=-60),
    "Sobreexpuesta (+70)":  variar_imagen(base, brillo=+70),
    "Con ruido (σ=25)":     variar_imagen(base, ruido_std=25),
    "Bajo contraste (0.6)": variar_imagen(base, contraste=0.6),
}

# Distancia L1 normalizada (MAE pixel a pixel)
def distancia_pixel(a, b):
    return np.mean(np.abs(a.astype(float) - b.astype(float)))

fig, axes = plt.subplots(1, len(variaciones), figsize=(14, 3), dpi=300)
for ax, (nombre, var) in zip(axes, variaciones.items()):
    ax.imshow(var, interpolation="nearest")
    d = distancia_pixel(base, var)
    ax.set_title(f"{nombre}\nd={d:.1f}", fontsize=8)
    ax.axis("off")

fig.suptitle("Misma célula, distintas condiciones de captura", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Comparamos distancias: misma célula vs. célula diferente
np.random.seed(0)

# Seleccionar 30 instancias de distintas clases
otras_celulas = [imagenes[np.where(etiquetas == c)[0][0]] for c in range(8)]
otras_celulas += [imagenes[np.where(etiquetas == c)[0][1]] for c in range(8)]

print("Distancias pixel a pixel:")
print("=" * 75)
print(f"  Original ↔ Original           : {distancia_pixel(base, base):.2f}  ← mínimo posible")

for nombre, var in list(variaciones.items())[1:]:
    d = distancia_pixel(base, var)
    barra = "█" * int(d / 3)
    print(f"  Original ↔ {nombre:<22}: {d:.2f}  {barra}")

print()
dists_otras = [distancia_pixel(base, c) for c in otras_celulas]
print(f"  Original ↔ célula diferente   : {np.mean(dists_otras):.2f} (promedio de 16 células)")
print()
print("⚠️  Una célula oscurecida puede estar MÁS alejada (en píxeles)")
print("   de sí misma que de otra célula completamente diferente.")
print("   → Necesitamos una representación más robusta.")

---
## **Sección 4** — Filtros y convoluciones clásicas

Antes del deep learning, la solución era **diseñar características a mano**: aplicar filtros matemáticos que detecten bordes, texturas, formas. Esto se llama *feature engineering*.

Una **convolución** aplica un filtro (kernel) $K$ deslizándolo sobre la imagen:

$$
(I * K)[i,j] = \sum_{m}\sum_{n} I[i+m,\, j+n] \cdot K[m,n]
$$

El resultado en cada posición dice *qué tanto se parece esa región al patrón del filtro*.

In [ ]:
# Kernels clásicos

# Trabajaremos en escala de grises para simplificar la visualización
img_gray = skcolor.rgb2gray(img_float)   # shape: (28, 28), rango [0, 1]

kernels = {
    "Original\n(sin filtro)": None,
    "Blur Gaussiano\n(suavizar)": np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1]
    ], dtype=float) / 16,
    "Sobel H\n(bordes horiz.)": np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1]
    ], dtype=float),
    "Sobel V\n(bordes vert.)": np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=float),
    "Laplaciano\n(todos los bordes)": np.array([
        [ 0, -1,  0],
        [-1,  4, -1],
        [ 0, -1,  0]
    ], dtype=float),
    "Sharpen\n(realzar detalles)": np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0]
    ], dtype=float),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=300)

for ax, (nombre, K) in zip(axes.flat, kernels.items()):
    if K is None:
        resultado = img_gray
    else:
        resultado = convolve(img_gray, K)
        resultado = np.clip(resultado, 0, 1)

    im = ax.imshow(resultado, cmap="gray", interpolation="nearest")
    ax.set_title(nombre, fontsize=9)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Mostrar el kernel en un recuadro pequeño
    if K is not None:
        ax_ins = ax.inset_axes([0.0, 0.0, 0.30, 0.30])
        ax_ins.imshow(K, cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
        ax_ins.set_xticks([])
        ax_ins.set_yticks([])
        ax_ins.set_title("K", fontsize=6, color="white", pad=1)

fig.suptitle(f"Filtros clásicos — {NOMBRES_CLASE[6]} (escala de grises)", fontsize=11)
plt.tight_layout()
plt.show()

Exploremos ahora el caso de las características extraídas con HOG.

In [ ]:
# HOG — Histogram of Oriented Gradients

# HOG no es un solo filtro: describe la DISTRIBUCIÓN de orientaciones de borde
# en celdas locales. Es más robusto a cambios de iluminación que comparar píxeles.

# Necesitamos imagen más grande para que HOG tenga sentido (28px es pequeño)
# Reescalamos a 112×112
img_grande = np.array(Image.fromarray(img).resize((112, 112)))
img_gray_grande = skcolor.rgb2gray(img_grande.astype(np.float64) / 255.0)

descriptor_hog, imagen_hog = skfeature.hog(
    img_gray_grande,
    orientations=9,       # Bins del histograma de ángulos
    pixels_per_cell=(14, 14),  # Tamaño de cada celda
    cells_per_block=(2, 2),    # Normalización por bloque
    visualize=True,
    feature_vector=True
)

fig, axes = plt.subplots(1, 3, figsize=(11, 4), dpi=300)

axes[0].imshow(img_grande, interpolation="nearest")
axes[0].set_title("Original (112×112)", fontsize=9)
axes[0].axis("off")

axes[1].imshow(img_gray_grande, cmap="gray", interpolation="nearest")
axes[1].set_title("Escala de grises", fontsize=9)
axes[1].axis("off")

axes[2].imshow(imagen_hog, cmap="inferno", interpolation="nearest")
axes[2].set_title("Visualización HOG", fontsize=9)
axes[2].axis("off")

fig.suptitle(f"HOG — {NOMBRES_CLASE[6]}", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Dimensión del descriptor HOG: {descriptor_hog.shape[0]} números")
print(f"(vs. {112*112*3} píxeles de la imagen original)")
print()
print("Las flechas en la visualización muestran la orientación dominante")
print("de los bordes en cada celda — una forma de codificar la FORMA de la célula.")

### 🏋️ **Ejercicio 3** — ¿Qué filtro resalta mejor el borde del núcleo?

Cambia `clase_id` para explorar distintas células y compara cuál de los filtros clásicos resalta mejor la diferencia entre el núcleo y el citoplasma.

In [ ]:
# Ejercicio 3: Comparar filtros en distintas clases
clase_id  = 1   # Cambia (0-7)
instancia = 1   # Cambia para ver otra célula de la misma clase

idx_ej3  = np.where(etiquetas == clase_id)[0][instancia]
img_ej3  = imagenes[idx_ej3].astype(np.float64) / 255.0
gray_ej3 = skcolor.rgb2gray(img_ej3)

filtros_ej = {
    "Original": None,
    "Sobel (magnitud)": "sobel",
    "Laplaciano":       np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=float),
    "Sharpen":          np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float),
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), dpi=300)
for ax, (nombre, filt) in zip(axes, filtros_ej.items()):
    if filt is None:
        res = img_ej3
        ax.imshow(res, interpolation="nearest")
    elif isinstance(filt, str) and filt == "sobel":
        res = skfilters.sobel(gray_ej3)
        ax.imshow(res, cmap="gray", interpolation="nearest")
    else:
        res = np.clip(convolve(gray_ej3, filt), 0, 1)
        ax.imshow(res, cmap="gray", interpolation="nearest")
    ax.set_title(nombre, fontsize=9)
    ax.axis("off")

fig.suptitle(f"Ejercicio 3 — {NOMBRES_CLASE[clase_id]} (instancia {instancia})", y=1.02)
plt.tight_layout()
plt.show()

---
## **Sección 5** — Embeddings: Una representación aprendida

Los filtros clásicos son útiles, pero requieren que un experto decida cuáles usar. Las **redes neuronales convolucionales** aprenden automáticamente qué filtros son útiles para la tarea.

El resultado final de una CNN (antes de la capa de clasificación) es un **embedding**: un vector compacto que resume el contenido visual de la imagen.

$$
\text{Imagen} \xrightarrow{\text{CNN}} \mathbf{z} \in \mathbb{R}^d
$$

**Propiedad clave:** imágenes similares → vectores cercanos en $\mathbb{R}^d$.

In [ ]:
# Cargar EfficientNet-B0 pre-entrenado

# num_classes=0 elimina la capa clasificadora → devuelve el embedding directamente
device = "cuda" if torch.cuda.is_available() else "cpu"

modelo = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
modelo = modelo.to(device).eval()

# Pipeline de preprocesamiento (debe coincidir con el entrenamiento original)
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std= [0.229, 0.224, 0.225]),
])

# Verificar dimensión del embedding
with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224).to(device)
    dim_emb = modelo(dummy).shape[1]

print(f"Dispositivo : {device}")
print(f"Modelo      : EfficientNet-B0")
print(f"Parámetros  : {sum(p.numel() for p in modelo.parameters()):,}")
print(f"Dim embedding: {dim_emb}")

In [ ]:
# Extraer el embedding de una sola célula
def imagen_a_embedding(img_array_uint8):
    """Convierte una imagen uint8 (H,W,3) en su embedding normalizado."""
    pil   = Image.fromarray(img_array_uint8)
    tensor = transform(pil).unsqueeze(0).to(device)   # (1, 3, 224, 224)
    with torch.no_grad():
        emb = modelo(tensor)                           # (1, 1280)
    emb = emb.squeeze(0).cpu().numpy()                 # (1280,)
    return emb / np.linalg.norm(emb)                   # normalizar a esfera unitaria

# Probar con nuestro neutrófilo
emb_ejemplo = imagen_a_embedding(base)

print(f"Shape del embedding : {emb_ejemplo.shape}")
print(f"Norma (debe ser 1.0): {np.linalg.norm(emb_ejemplo):.6f}")
print(f"Min / Max           : {emb_ejemplo.min():.4f} / {emb_ejemplo.max():.4f}")
print()
print("Primeros 10 valores del embedding:")
print(np.round(emb_ejemplo[:10], 4))
print()
print(f"Compresión: {base.size} px → {emb_ejemplo.shape[0]} números ({base.size/emb_ejemplo.shape[0]:.0f}x)")

Podemos visualizar la imagen original:

In [ ]:
base

Podríamos desplegar el embedding:

In [ ]:
emb_ejemplo

In [ ]:
# Robustez del embedding vs. comparación en píxeles
def similitud_coseno(a, b):
    """Para vectores normalizados: producto punto = similitud coseno."""
    return float(np.dot(a, b))

print("Comparación: misma célula con variaciones vs. célula diferente")
print("=" * 62)
print(f"  {"Variación":<28}  {"dist. píxel":>12}  {"sim. coseno":>12}")
print("-" * 62)

emb_base = imagen_a_embedding(base)

for nombre, var in variaciones.items():
    d_px  = distancia_pixel(base, var)
    emb_v = imagen_a_embedding(var)
    s_cos = similitud_coseno(emb_base, emb_v)
    print(f"  {nombre:<28}  {d_px:>12.2f}  {s_cos:>12.4f}")

print("-" * 62)
# Célula completamente diferente (eosinófilo)
celula_dif = imagenes[np.where(etiquetas == 1)[0][0]]
d_px_dif   = distancia_pixel(base, celula_dif)
s_cos_dif  = similitud_coseno(emb_base, imagen_a_embedding(celula_dif))
print(f"  {"Eosinófilo (clase diferente)":<28}  {d_px_dif:>12.2f}  {s_cos_dif:>12.4f}")
print()
print("💡 Observa: la distancia en píxeles varía MUCHO con el brillo,")
print("   pero la similitud coseno entre embeddings permanece ALTA")
print("   para la misma célula y BAJA para la célula diferente.")

In [ ]:
# Extraer embeddings de 5 imágenes por clase

# En total: 8 clases × 5 imágenes = 40 embeddings
N_POR_CLASE = 5
print(f"Extrayendo {N_POR_CLASE} embeddings por clase (total: {N_POR_CLASE * 8})...")

embs_todos   = []
labels_todos = []

for clase in range(8):
    indices_clase = np.where(etiquetas == clase)[0][:N_POR_CLASE]
    for idx_c in indices_clase:
        emb = imagen_a_embedding(imagenes[idx_c])
        embs_todos.append(emb)
        labels_todos.append(clase)
    print(f"  Clase {clase} ({NOMBRES_CLASE[clase]:<16}): listo")

embs_todos   = np.array(embs_todos)    # (40, 1280)
labels_todos = np.array(labels_todos)  # (40,)
print(f"\nShape final: {embs_todos.shape}")

In [ ]:
# Matriz de similitud coseno

# sim[i, j] = cuán similares son la imagen i y la imagen j en el espacio de embeddings
matriz_sim = embs_todos @ embs_todos.T   # (40, 40)

# Etiquetas cortas para el eje
etiquetas_cortas = [
    f"{NOMBRES_CLASE[l][:4]}.{i%N_POR_CLASE}"
    for i, l in enumerate(labels_todos)
]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matriz_sim, cmap="viridis", vmin=0.5, vmax=1.0, aspect="auto")
plt.colorbar(im, ax=ax, label="Similitud coseno", fraction=0.03)

ax.set_xticks(range(len(etiquetas_cortas)))
ax.set_yticks(range(len(etiquetas_cortas)))
ax.set_xticklabels(etiquetas_cortas, rotation=90, fontsize=7)
ax.set_yticklabels(etiquetas_cortas, fontsize=7)

# Líneas divisorias entre clases
for i in range(1, 8):
    pos = i * N_POR_CLASE - 0.5
    ax.axhline(pos, color="white", lw=1.5, alpha=0.8)
    ax.axvline(pos, color="white", lw=1.5, alpha=0.8)

# Etiquetas de clase en el centro de cada bloque
for clase in range(8):
    centro = clase * N_POR_CLASE + N_POR_CLASE / 2 - 0.5
    ax.text(centro, -2.5, NOMBRES_CLASE[clase],
            ha="center", va="bottom", fontsize=7.5,
            color="#aaa", rotation=30)

ax.set_title("Matriz de similitud coseno entre embeddings\n"
             "(verde-amarillo = similar, morado = distinto)",
             fontsize=11, pad=20)
plt.tight_layout()
plt.show()

# Estadísticas
sims_intra = []
sims_inter = []
for i in range(len(labels_todos)):
    for j in range(i+1, len(labels_todos)):
        if labels_todos[i] == labels_todos[j]:
            sims_intra.append(matriz_sim[i, j])
        else:
            sims_inter.append(matriz_sim[i, j])

print(f"Similitud media INTRA-clase (misma célula)   : {np.mean(sims_intra):.4f}")
print(f"Similitud media INTER-clase (células distintas): {np.mean(sims_inter):.4f}")
print()
print("💡 Los bloques diagonales (misma clase) son más verdes.")
print("   Esto indica que el embedding agrupa células del mismo tipo.")
print("   → Con este espacio, podemos hacer búsqueda por similitud visual.")

---
## **Resumen del Notebook 1**

| Concepto | Descripción |
|----------|-------------|
| **Imagen = matriz** | `shape=(28,28,3)`, `dtype=uint8`, valores en `[0,255]` |
| **Canales RGB** | Cada canal captura una componente del color; B captura bien el núcleo teñido |
| **Espacios de color** | HSV separa tono del brillo; L\* de LAB facilita segmentación por luminosidad |
| **Variabilidad** | Distancia en píxeles es frágil ante cambios de iluminación |
| **Filtros clásicos** | Convoluciones detectan bordes/texturas; HOG codifica forma local |
| **Embedding** | CNN comprime imagen a vector de 1280 dims más robusto que los píxeles |
| **Similitud coseno** | Células del mismo tipo → embeddings más cercanos entre sí |

**A continuación...** `[BeePy 5.0] Notebook 2 - Clasificación con CNNs e Interpretabilidad con GradCAM` — clasificar células con EfficientNet y visualizar con GradCAM qué regiones activa la red.

---
> Contenido por **Rodolfo Ferro**. Contacto: [ferro@cimat.mx](ferro@cimat.mx) <br>
BeePy 5.0, 2026.